<a href="https://colab.research.google.com/github/Shreyas-Reddy707/Computer-Vision/blob/main/Computer_Visions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study: Quantitative Impact of Image Denoising Prior to Edge Detection

This notebook demonstrates and quantifies how pre-processing (denoising/blurring) an image affects the quality and accuracy of edge-detection algorithms, specifically the **Canny Edge Detector**.

## 📌 Project Overview
Edge detection is highly sensitive to high-frequency noise. Without proper filtering, noise is often misinterpreted as legitimate structural boundaries, resulting in massive false-positive rates. Conversely, over-filtering can blur actual structural transitions, resulting in false negatives.

### 🔧 Core Workflow
1. **Convert Input Image:** Standardize any color input to grayscale.
2. **Ground Truth Extraction:** Compute Canny edges directly on the original clean image.
3. **Noise Injection:** Inject adjustable Gaussian noise to create a realistic noisy input.
4. **Control Run (No Denoising):** Run Canny edge detection on the raw noisy image.
5. **Denoising Step:** Smooth the noisy image using a Gaussian filter.
6. **Experimental Run (With Denoising):** Run Canny edge detection on the filtered image.
7. **Quantitative Evaluation:** Measure Structural Similarity (SSIM), Precision, and F1-score against the original clean ground truth.
8. **Visual Validation:** Render a color-coded structural classification overlay map (TP, FP, FN).

In [1]:
!pip install gradio scikit-image -q

import cv2
import numpy as np
import pandas as pd
import gradio as gr
from skimage.metrics import structural_similarity as ssim

def process_image(image, noise_intensity, blur_kernel_size):
    if image is None:
        empty_table = pd.DataFrame(columns=["Metric", "Without Denoising", "With Denoising"])
        return None, None, None, None, empty_table

    # 1. Convert to grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image.copy()

    # 2. Calculate Ground Truth edges from clean grayscale image
    edges_gt = cv2.Canny(gray, 50, 150)
    edges_gt_bin = (edges_gt > 0).astype(bool)

    # 3. Inject synthetic Gaussian noise
    h, w = gray.shape
    std = noise_intensity * 255
    noise = np.random.normal(0, std, (h, w))
    noisy_image = np.clip(gray.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # 4. Compute Canny edges directly on the noisy image (Control)
    edges_noisy = cv2.Canny(noisy_image, 50, 150)
    edges_noisy_bin = (edges_noisy > 0).astype(bool)

    # 5. Apply Gaussian blur to the noisy image (Result)
    ksize = int(blur_kernel_size)
    if ksize % 2 == 0:
        ksize += 1
    blurred = cv2.GaussianBlur(noisy_image, (ksize, ksize), 0)

    # 6. Compute Canny edges on the blurred image
    edges_denoised = cv2.Canny(blurred, 50, 150)
    edges_denoised_bin = (edges_denoised > 0).astype(bool)

    # 7. Compute Structural Similarity Index (SSIM) between processed and original grayscale
    ssim_noisy = ssim(gray, noisy_image)
    ssim_denoised = ssim(gray, blurred)

    # Helper to calculate edge classification and metrics
    def get_edge_metrics(pred_bin, gt_bin):
        tp = np.sum(pred_bin & gt_bin)
        fp = np.sum(pred_bin & ~gt_bin)
        fn = np.sum(~pred_bin & gt_bin)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        return fp, precision, f1, tp, fn

    fp_noisy, prec_noisy, f1_noisy, _, _ = get_edge_metrics(edges_noisy_bin, edges_gt_bin)
    fp_denoised, prec_denoised, f1_denoised, tp_denoised, fn_denoised = get_edge_metrics(edges_denoised_bin, edges_gt_bin)

    # 8. Create dynamic, high-visibility Color Overlay for the Denoised result
    # Reduce the background intensity to 30% brightness so the colorful edges pop
    background_dimmed = (gray.astype(np.float32) * 0.3).astype(np.uint8)
    overlay = cv2.cvtColor(background_dimmed, cv2.COLOR_GRAY2RGB)

    # Separate logical maps
    tp_mask = edges_denoised_bin & edges_gt_bin
    fp_mask = edges_denoised_bin & ~edges_gt_bin
    fn_mask = ~edges_denoised_bin & edges_gt_bin

    # Dilate masks slightly (makes the 1-pixel edges thicker and 3x more visible)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

    dilated_tp = cv2.dilate(tp_mask.astype(np.uint8), kernel).astype(bool)
    dilated_fp = cv2.dilate(fp_mask.astype(np.uint8), kernel).astype(bool)
    dilated_fn = cv2.dilate(fn_mask.astype(np.uint8), kernel).astype(bool)

    # Map colors with priority order: Blue (FN) -> Red (FP) -> Green (TP) on top
    overlay[dilated_fn] = [0, 0, 255]    # Blue (FN)
    overlay[dilated_fp] = [255, 0, 0]    # Red (FP)
    overlay[dilated_tp] = [0, 255, 0]    # Green (TP)

    # 9. Format Benchmarking Metrics Table
    metrics_data = {
        "Metric": [
            "False-Positive Edge Pixels",
            "Edge Precision (TP / (TP + FP))",
            "F1-Score",
            "Image SSIM (relative to Clean Ground Truth)"
        ],
        "Direct Canny (No Denoise)": [
            f"{fp_noisy:,}",
            f"{prec_noisy:.4f}",
            f"{f1_noisy:.4f}",
            f"{ssim_noisy:.4f}"
        ],
        "Filtered Canny (With Denoising)": [
            f"{fp_denoised:,}",
            f"{prec_denoised:.4f}",
            f"{f1_denoised:.4f}",
            f"{ssim_denoised:.4f}"
        ]
    }
    metrics_df = pd.DataFrame(metrics_data)

    return noisy_image, edges_noisy, edges_denoised, overlay, metrics_df

# Building the UI using Gradio Blocks
with gr.Blocks() as demo:
    gr.Markdown(
        """
        # Case Study: Quantitative Impact of Denoising Prior to Edge Detection
        Compare edge-detection results against the original, clean image ground truth. Adjust sliders to observe changes in image similarity (SSIM), Canny precision, and structural classification overlays in real time.
        """
    )

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Upload Image", type="numpy")
            noise_slider = gr.Slider(minimum=0.0, maximum=0.5, step=0.05, value=0.15, label="Noise Intensity")
            kernel_slider = gr.Slider(minimum=3, maximum=15, step=2, value=5, label="Blur Kernel Size")

    with gr.Row():
        out_noisy = gr.Image(label="Noisy Image", interactive=False)
        out_edges_noisy = gr.Image(label="Edges WITHOUT Denoising", interactive=False)
        out_edges_denoised = gr.Image(label="Edges WITH Denoising", interactive=False)

    with gr.Row():
        with gr.Column(scale=1):
            out_overlay = gr.Image(label="High-Visibility Edge Validation (Green: TP, Red: FP, Blue: FN)", interactive=False)
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Edge Detection Performance Metrics Table")
            metrics_table = gr.Dataframe(
                headers=["Metric", "Direct Canny (No Denoise)", "Filtered Canny (With Denoising)"],
                datatype=["str", "str", "str"],
                interactive=False
            )

    gr.Markdown(
        """
        ### 📝 Case Study Metrics Guide
        - **True Positives (Green):** Edges accurately detected on both the clean and denoised image.
        - **False Positives (Red):** Phantom edges caused by high-frequency background noise triggers.
        - **False Negatives (Blue):** Legitimate boundaries that were blurred out of visibility due to excessive filtering.
        """
    )

    inputs = [input_image, noise_slider, kernel_slider]
    outputs = [out_noisy, out_edges_noisy, out_edges_denoised, out_overlay, metrics_table]

    input_image.change(fn=process_image, inputs=inputs, outputs=outputs)
    noise_slider.change(fn=process_image, inputs=inputs, outputs=outputs)
    kernel_slider.change(fn=process_image, inputs=inputs, outputs=outputs)

# Launch app externally
def launch_demo():
    demo.launch(share=True, inline=False, theme=gr.themes.Monochrome())

launch_demo()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://68e2e9f072d442910b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 📊 Detailed Visual & Statistical Metrics Guide

To help visualize where the edge detector succeeds and fails, the dynamic visualization overlay classifies every single edge pixel into three distinct categories:

*   <span style="color:green">**True Positives (Green):**</span> Correctly identified edge boundaries.
*   <span style="color:red">**False Positives (Red):**</span> Phantom edges detected due to background Gaussian noise.
*   <span style="color:blue">**False Negatives (Blue):**</span> Actual edge boundaries that were missed (often due to over-smoothing from large kernel sizes).

### 🚀 Getting Started
1. Run the Gradio cell above.
2. Click the temporary public link or use the inline interface to upload an image.
3. Adjust the **Noise Intensity** and **Blur Kernel Size** sliders in real-time to find the optimal balance that maximizes the F1-Score!